# 44 — MCP Client Agent

## What you'll learn

- **MCP tool discovery vs hardcoded tools** — Why dynamic discovery matters for composable agents
- **`list_tools()` + `call_tool()` contract** — The two primitives that define the MCP client interface
- **JSON tool selection** — How an LLM picks a tool by name and constructs arguments from a description
- **Mock-to-real migration path** — How to swap `MockMCPServer` for `langchain-mcp-adapters` with a real server

## Context

**MCP (Model Context Protocol)** is an open standard introduced by Anthropic in 2024 that defines how AI agents
communicate with external tool servers. Instead of hardcoding tool implementations inside the agent,
MCP separates concerns: the **server** owns tool logic; the **client** (the agent) discovers and invokes
tools at runtime over a transport (stdio, HTTP+SSE). This notebook demonstrates the core pattern
using an in-process mock — no subprocess, no network required.

In [ ]:
# Install dependencies (safe to re-run)
import subprocess, sys
pkgs = ["langchain", "langchain-openai", "langgraph", "python-dotenv"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
print("Dependencies ready")

In [ ]:
import os

# Colab: uncomment and paste your key
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Local: loads from .env file
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"
print("API key loaded")

In [ ]:
# MCP Architecture — Three key ideas
#
#  +---------------+   list_tools()    +--------------------+
#  |  MCP Client   | ----------------> |    MCP Server      |
#  |  (Agent)      | <----------------  |  (Tool Registry)   |
#  |               |   [{name, desc}]  |                    |
#  |               |                   |  - get_weather      |
#  |               |   call_tool(      |  - calculate_tip   |
#  |               |     name, args)   |  - get_time        |
#  |               | ----------------> |                    |
#  |               | <----------------  |                    |
#  +---------------+    result str     +--------------------+
#
# Idea 1: Tools are DISCOVERED at runtime, not hardcoded in the agent
# Idea 2: The SERVER owns tool implementations; client only knows the interface
# Idea 3: The contract is simple: list_tools() -> call_tool(name, args) -> str

print("MCP pattern: discover -> select -> invoke -> synthesize")

In [ ]:
import json
from dataclasses import dataclass
from typing import Callable, TypedDict

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph

# -- State --------------------------------------------------------------------

class MCPState(TypedDict):
    query: str
    available_tools: list[dict]
    tool_name: str
    tool_args: dict
    tool_result: str
    final_answer: str

# -- Mock MCP Server ----------------------------------------------------------

@dataclass
class MCPTool:
    name: str
    description: str
    fn: Callable


class MockMCPServer:
    """Simulates an MCP server's tool registry (in-process, no subprocess)."""

    def __init__(self):
        self._tools: dict[str, MCPTool] = {}

    def register(self, name: str, description: str, fn: Callable):
        self._tools[name] = MCPTool(name=name, description=description, fn=fn)

    def list_tools(self) -> list[dict]:
        """MCP contract: returns [{name, description}, ...]"""
        return [{"name": t.name, "description": t.description} for t in self._tools.values()]

    def call_tool(self, name: str, args: dict) -> str:
        """MCP contract: invokes tool by name, returns result as string."""
        if name not in self._tools:
            return f"Unknown tool: {name}"
        return str(self._tools[name].fn(**args))


def build_mock_server() -> MockMCPServer:
    server = MockMCPServer()
    server.register("get_weather", "Get current weather for a city",
                    lambda city: f"Sunny, 22 deg C in {city}")
    server.register("calculate_tip", "Calculate tip amount",
                    lambda bill, pct: f"${bill * pct / 100:.2f} tip")
    server.register("get_time", "Get current time in a timezone",
                    lambda city: f"12:34 PM in {city}")
    return server


# -- LLM + Prompts ------------------------------------------------------------

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
_server = build_mock_server()

DISCOVER_PROMPT = """You have access to these tools from an MCP server:
{tools}

For this query: "{query}"
Which tool should be called? Respond as JSON: {{"tool": "tool_name", "args": {{...}}}}
If no tool fits, respond: {{"tool": "none", "args": {{}}}}"""

ANSWER_PROMPT = """Query: {query}
Tool called: {tool_name}
Tool result: {tool_result}
Provide a helpful final answer."""

# -- Graph Nodes --------------------------------------------------------------

def discover_and_select(state: MCPState) -> dict:
    tools = _server.list_tools()
    tools_text = "\n".join(f"- {t['name']}: {t['description']}" for t in tools)
    prompt = DISCOVER_PROMPT.format(tools=tools_text, query=state["query"])
    result = llm.invoke([HumanMessage(content=prompt)])
    try:
        text = result.content.strip()
        if text.startswith("```"):
            text = "\n".join(text.split("\n")[1:-1])
        parsed = json.loads(text)
        print(f"  Selected: {parsed.get('tool')} with args {parsed.get('args')}")
        return {"available_tools": tools, "tool_name": parsed.get("tool", "none"),
                "tool_args": parsed.get("args", {})}
    except Exception:
        return {"available_tools": tools, "tool_name": "none", "tool_args": {}}


def invoke_tool(state: MCPState) -> dict:
    if state["tool_name"] == "none":
        return {"tool_result": "No tool selected"}
    result = _server.call_tool(state["tool_name"], state["tool_args"])
    print(f"  Tool result: {result}")
    return {"tool_result": result}


def synthesize(state: MCPState) -> dict:
    prompt = ANSWER_PROMPT.format(
        query=state["query"], tool_name=state["tool_name"], tool_result=state["tool_result"]
    )
    result = llm.invoke([HumanMessage(content=prompt)])
    return {"final_answer": result.content}


# -- Build Graph --------------------------------------------------------------

def create_workflow():
    graph = StateGraph(MCPState)
    graph.add_node("discover", discover_and_select)
    graph.add_node("invoke", invoke_tool)
    graph.add_node("synthesize", synthesize)
    graph.set_entry_point("discover")
    graph.add_edge("discover", "invoke")
    graph.add_edge("invoke", "synthesize")
    graph.add_edge("synthesize", END)
    return graph.compile()


app = create_workflow()
print("Workflow compiled -- nodes: discover -> invoke -> synthesize -> END")

In [ ]:
SAMPLE_QUERIES = [
    "What is the weather like in Tokyo?",
    "Calculate 15% tip on a $47.50 bill.",
    "What time is it in London right now?",
]

for query in SAMPLE_QUERIES:
    print(f"\nQUERY: {query}")
    result = app.invoke({
        "query": query, "available_tools": [], "tool_name": "",
        "tool_args": {}, "tool_result": "", "final_answer": "",
    })
    print(f"Tool: {result['tool_name']} -> {result['tool_result']}")
    print(f"Answer: {result['final_answer'][:200]}")

## Exercises

1. **Add a new tool**: Register `convert_currency` on `MockMCPServer` (e.g., `lambda amount, from_curr, to_curr: f"{amount} {from_curr} = {amount * 1.08} {to_curr}"`). Add a matching query to `SAMPLE_QUERIES` and verify the agent selects it.

2. **Real MCP server**: Replace `MockMCPServer` with `langchain-mcp-adapters` connected to a real MCP server (e.g., `npx -y @modelcontextprotocol/server-filesystem`). The `list_tools()` / `call_tool()` contract is identical — only the instantiation changes.

3. **Argument validation**: What happens if the LLM passes args of the wrong type to `call_tool()`? Add a try/except in `invoke_tool` that returns a descriptive error string and re-runs discovery. Observe how the agent recovers.

4. **Tool-not-found handling**: Force `tool_name = "unknown_tool"` and trace what `MockMCPServer.call_tool` returns. Extend the workflow with a fallback node that generates a direct LLM answer when tool invocation fails.

## What's next

- **`4-basic-react-agent`** — Hardcoded tool binding with `bind_tools`; contrast with dynamic MCP discovery
- **`6-multi-agent-graph`** — Route queries to specialist sub-agents; MCP servers can serve as agent boundaries